# 04. Huấn luyện nhóm mô hình cây 

## Mục tiêu:
* Triển khai các thuật toán dựa trên cấu trúc cây: **Decision Tree**, **Random Forest**, và **Gradient Boosting (XGBoost/LightGBM)**.
* Thực hiện **Hyperparameter Tuning** để tìm bộ tham số tối ưu cho bài toán dự đoán tuổi bào ngư (Abalone Age).
* Phân tích mức độ quan trọng của các đặc trưng (**Feature Importance**) để hiểu rõ các yếu tố ảnh hưởng đến biến mục tiêu `Rings`.
* Đánh giá và so sánh hiệu suất giữa các mô hình trong nhóm Tree.

In [11]:
import sys
import warnings
import time
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Mô hình nhóm Cây (Tree-based Models)
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Công cụ đánh giá và Tuning
from sklearn.model_selection import GridSearchCV, cross_val_score, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Quản lý cấu trúc thư mục (Project Root)
# File này nằm trong notebooks/research/ nên cần lùi 2 cấp để về gốc AbaloneAge/
current_path = Path.cwd()
PROJECT_ROOT = current_path.parent.parent if current_path.name == 'research' else current_path.resolve()

# Thêm PROJECT_ROOT vào sys.path để gọi được các hàm trong src/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2. Định nghĩa các đường dẫn thư mục chức năng
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR    = PROJECT_ROOT / 'outputs' / 'models'
METRICS_DIR   = PROJECT_ROOT / 'outputs' / 'metrics'
FIGURES_DIR   = PROJECT_ROOT / 'outputs' / 'figures'

# Đảm bảo các thư mục output tồn tại
for folder in [MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# 3. Import các hàm Helper từ thư mục src/ của nhóm
from src.models.evaluate import evaluate_regression_metrics
from src.visualization.plot_results import plot_prediction_scatter, plot_residual_scatter

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [13]:
# 1. Định nghĩa tên các file dữ liệu
TRAIN_FILE = 'abalone_train_standard.csv'
TEST_FILE  = 'abalone_test_standard.csv'

# 2. Nạp dữ liệu từ PROCESSED_DIR
train_df = pd.read_csv(PROCESSED_DIR / TRAIN_FILE)
test_df  = pd.read_csv(PROCESSED_DIR / TEST_FILE)
    
# Tách Features (X) và Target (y) - Cột mục tiêu là 'Rings'
X_train = train_df.drop(columns=['Rings'])
y_train = train_df['Rings'].values
    
X_test  = test_df.drop(columns=['Rings'])
y_test  = test_df['Rings'].values
    
print(f"   - Tập Huấn luyện (Train): {X_train.shape}")
print(f"   - Tập Kiểm thử (Test):    {X_test.shape}")
    
# Hiển thị 5 dòng đầu của tập Train để kiểm tra các cột sau Encoding
display(X_train.head())

   - Tập Huấn luyện (Train): (2923, 10)
   - Tập Kiểm thử (Test):    (1254, 10)


,Sex_F,Sex_I,Sex_M,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
0,1.0,0.0,0.0,-0.009546,0.206802,-0.120724,0.014578,0.311054,-0.020982,-0.425408
1,0.0,1.0,0.0,-0.803180,-0.854049,-0.944651,-0.959009,-0.919416,-0.913759,-0.969745
2,0.0,0.0,1.0,-0.594329,-0.601465,-0.826947,-0.854333,-0.897043,-0.780987,-0.685124
3,0.0,0.0,1.0,-2.682841,-2.571616,-2.239393,-1.613488,-1.548074,-1.618824,-1.606585
4,1.0,0.0,0.0,0.533467,0.560419,0.467795,0.536941,0.639926,0.642877,0.382204


## 2. Huấn luyện Baseline

In [ ]:
# 1. Khởi tạo danh sách lưu kết quả
baseline_results = []

# 2. Định nghĩa danh sách mô hình
tree_models = {
    "Decision_Tree": DecisionTreeRegressor(random_state=42),
    "Random_Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "Gradient_Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42)
}

print("--- Đang huấn luyện Baseline ---")

for name, model in tree_models.items():
    try:
        start_t = time.time()
        model.fit(X_train, y_train)
        duration = time.time() - start_t
        
        y_pred = model.predict(X_test)
        
        # Lấy kết quả (MAE, MSE, RMSE, RSE)
        res = evaluate_regression_metrics(y_test, y_pred)
        
        # Copy kết quả và thêm tên model
        row = res.copy()
        row['Model'] = name
        row['Train_Time'] = duration
        
        baseline_results.append(row)
        print(f"Xong: {name}")
        
    except Exception as e:
        print(f"Lỗi tại {name}: {e}")

# 3. Chuyển sang DataFrame
df_baseline = pd.DataFrame(baseline_results)

# 4. Sắp xếp theo RMSE (Vì nhóm bạn không có R2, ta dùng RMSE - Càng thấp càng tốt)
if 'RMSE' in df_baseline.columns:
    df_baseline = df_baseline.sort_values(by='RMSE', ascending=True)
    print("\n--- Bảng kết quả sắp xếp theo RMSE (Thấp nhất dẫn đầu) ---")
else:
    print("\nCác cột hiện có:", df_baseline.columns.tolist())

display(df_baseline)

# 5. Lưu file
df_baseline.to_csv(METRICS_DIR / 'tree_baseline_results.csv', index=False)

--- 🚀 Đang huấn luyện Baseline ---
✅ Xong: Decision_Tree
✅ Xong: Random_Forest
✅ Xong: Gradient_Boosting
✅ Xong: XGBoost

❌ Các cột hiện có: ['mse', 'rmse', 'rse', 'mae', 'Model', 'Train_Time']


,mse,rmse,rse,mae,Model,Train_Time
0,8.816587,2.969274,0.868239,2.052632,Decision_Tree,0.021805
1,4.827886,2.197245,0.475440,1.545199,Random_Forest,0.168392
2,4.757438,2.181155,0.468503,1.542060,Gradient_Boosting,0.409821
3,5.514435,2.348283,0.543050,1.659860,XGBoost,0.100543
